In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import sys
import os
from pathlib import Path

In [12]:
print(sys.path)

['/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python311.zip', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11', '/opt/homebrew/Cellar/python@3.11/3.11.8/Frameworks/Python.framework/Versions/3.11/lib/python3.11/lib-dynload', '', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/src', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/de_etl_examples/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework', '/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src']


In [13]:
root_path = (Path(os.getcwd()).parent.parent / "src").as_posix()
root_path

'/Users/mike.sturwold/CodeProjects/ADM/dna-common-all-databricks-templates/Source/adm_integration_framework/src'

In [14]:
sys.path.append(root_path)

In [15]:
import unittest
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, Row, DecimalType
from adm.integration.adm_data_transformer import AdmDataTransformer
from adm.integration.adm_base_data_transformer import AdmBaseDataTransformer
from adm.integration.adm_table_writer import AdmTableWriter
from adm.integration.adm_table_reader import AdmTableReader
import logging

In [16]:
config_file = "use_cases/p2c_insightops/sales_order_invoice_jdexeeu.yml"
adm_table_reader = AdmTableReader(config_file)
transformer = AdmBaseDataTransformer(config_file)
transformer2 = AdmDataTransformer(config_file)
writer = AdmTableWriter(config_file)


In [17]:
df = transformer2.apply_joins("combine_1")

INFO:adm.integration.base_integration:Applying joins for combine_id: combine_1
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_order_detail
INFO:adm.integration.base_integration:Applying function Name: use_cases.custom_transformations.p2c_tx.convert_julian_dates
INFO:adm.integration.base_integration:Reading and selecting columns for source_id: jdexeeu_customer_ledger with alias: cl
INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_customer_ledger
INFO:adm.integration.base_integration:Applying function Name: use_cases.custom_transformations.p2c_tx.convert_julian_dates
INFO:adm.integration.base_integration:Joining od with cl using condition: od.invoice_number = cl.RPDOC  and od.document_type = cl.RPDCT  and od.SDKCO = cl.RPKCO  and od.SDDOCO = cl.RPSDOC  and od.SDLNID = cl.RPLNID and join type: inner
INFO:adm.integration.base_integration:Schema after joining od with cl: StructType([StructField('invoice_number', DoubleType(), Tru

In [19]:
df2 = transformer.apply_transformations("jdexeeu_joined", df)

INFO:adm.integration.base_integration:Starting transformations for ID: jdexeeu_joined


In [21]:
writer.write_table("sales_order_invoice_jdexeeu", df2)

INFO:adm.integration.base_integration:Number of records written to table sandbox.de_ref_silver_mikesturwoldcontractor.sales_order_invoice_jdexeeu: 3150468


In [22]:
display(df2.groupBy("__INSERTED_TIMESTAMP").count())

,__INSERTED_TIMESTAMP,count
0,2025-05-05 19:52:19.992,3150468
